In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    StackingRegressor,
    VotingRegressor,
    ExtraTreesRegressor
)
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib

SEED = 42
np.random.seed(SEED)

# Загружаем данные
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val = pd.read_csv('../data/processed/X_val.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_val = pd.read_csv('../data/processed/y_val.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Определяем числовые и категориальные колонки
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Препроцессор с обработкой пропусков
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

def get_pipeline(model):
    """Оборачивает модель в пайплайн с предобработкой"""
    return Pipeline([('prep', preprocessor), ('reg', model)])

# Словарь моделей (все из sklearn)
models = {
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(random_state=SEED),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
    'Ridge': Ridge(alpha=1.0, random_state=SEED),
    'Lasso': Lasso(alpha=0.01, random_state=SEED, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=SEED, max_iter=10000),
    'KNN': KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
    'MLP': MLPRegressor(hidden_layer_sizes=(100, 50), random_state=SEED, max_iter=500, early_stopping=True),
    'SVR': SVR(kernel='rbf', C=1.0, epsilon=0.1)  # может обучаться долго – закомментировать если медленно
}

# Обучаем каждую модель и сохраняем результат на валидации
results = {}
best_model = None
best_r2 = -np.inf

for name, model in models.items():
    print(f"Обучаем {name}...")
    pipe = get_pipeline(model)
    pipe.fit(X_train, y_train)
    y_pred_val = pipe.predict(X_val)
    r2 = r2_score(y_val, y_pred_val)
    results[name] = r2
    print(f"  → R² (val) = {r2:.4f}")
    # Сохраняем лучшую модель
    if r2 > best_r2:
        best_r2 = r2
        best_model = pipe
    # Сохраняем каждую модель в файл
    joblib.dump(pipe, f'../models/{name.lower()}_model.pkl')

print("\nРезультаты на валидации:")
for name, r2 in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name}: {r2:.4f}")

# ---- Ансамблирование ----
# 1. Voting Regressor (усреднение предсказаний лучших моделей)
# Выбираем топ-3 модели по R²
top_models = sorted(results.items(), key=lambda x: x[1], reverse=True)[:3]
estimators = [(name, models[name]) for name, _ in top_models]

voting = VotingRegressor(estimators=estimators, n_jobs=-1)
voting_pipe = Pipeline([('prep', preprocessor), ('voting', voting)])
voting_pipe.fit(X_train, y_train)
y_pred_voting = voting_pipe.predict(X_val)
r2_voting = r2_score(y_val, y_pred_voting)
print(f"\nVoting Regressor (top-3): R² = {r2_voting:.4f}")
joblib.dump(voting_pipe, '../models/voting_model.pkl')
if r2_voting > best_r2:
    best_r2 = r2_voting
    best_model = voting_pipe

# 2. Stacking Regressor (используем лучшие модели)
stack_estimators = [(name, models[name]) for name, _ in top_models]
stack = StackingRegressor(
    estimators=stack_estimators,
    final_estimator=Ridge(random_state=SEED),
    cv=3,
    n_jobs=-1
)
stack_pipe = Pipeline([('prep', preprocessor), ('stack', stack)])
stack_pipe.fit(X_train, y_train)
y_pred_stack = stack_pipe.predict(X_val)
r2_stack = r2_score(y_val, y_pred_stack)
print(f"Stacking Regressor: R² = {r2_stack:.4f}")
joblib.dump(stack_pipe, '../models/stacking_model.pkl')
if r2_stack > best_r2:
    best_r2 = r2_stack
    best_model = stack_pipe

print(f"\nЛучшая модель (по R² на валидации): R² = {best_r2:.4f}")
# Сохраняем лучшую модель отдельно
joblib.dump(best_model, '../models/best_model.pkl')

# ---- Тест на тестовой выборке для лучшей модели ----
y_pred_test = best_model.predict(X_test)
r2_test = r2_score(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
print(f"\nЛучшая модель на ТЕСТЕ: R² = {r2_test:.4f}, MAE = {mae_test:.2f}, RMSE = {rmse_test:.2f}")

/var/folders/t9/w50p8w0917b0t61ks_6y_3g40000gn/T/ipykernel_24907/234792265.py:34: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['order_time' 'hour' 'minute']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Обучаем RandomForest...


/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['order_time' 'hour' 'minute']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


  → R² (val) = 0.9498
Обучаем GradientBoosting...


/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['order_time' 'hour' 'minute']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['order_time' 'hour' 'minute']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/smerkalovaanasta

...

  → R² (val) = 0.9446

Результаты на валидации:
  GradientBoosting: 0.9533
  Lasso: 0.9501
  ElasticNet: 0.9500
  RandomForest: 0.9498
  SVR: 0.9446
  Ridge: 0.9436
  MLP: 0.9405
  ExtraTrees: 0.9395
  KNN: 0.8742


/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['order_time' 'hour' 'minute']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['order_time' 'hour' 'minute']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/smerkalovaanasta


Voting Regressor (top-3): R² = 0.9518


/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['order_time' 'hour' 'minute']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/smerkalovaanastasia/Desktop/starbucks/hseml-group-project-aasmerkalova/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['order_time' 'hour' 'minute']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/smerkalovaanasta

Stacking Regressor: R² = 0.9533

Лучшая модель (по R² на валидации): R² = 0.9533

Лучшая модель на ТЕСТЕ: R² = 0.9549, MAE = 0.97, RMSE = 1.18
